## QLoRA Fine-tuning Setup

In [1]:
# Install necessary libraries for QLoRA fine-tuning
!pip install -q transformers peft bitsandbytes accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00


### 1. Load the Model and Tokenizer

We'll load the `Qwen2.5-1.5B-Instruct` model and its tokenizer. For QLoRA, we load the model in 4-bit quantization using `BitsAndBytesConfig`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2-1.5B-Instruct"

# Configure 4-bit quantization
bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Changed to float16 to match fp16=True
    bnb_4bit_use_double_quant=True,
)

# Load the model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bitsandbytes_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Model and Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

### 2. Load and Prepare the Dataset

For demonstration, I'll load a small instruction-based dataset from Hugging Face (`HuggingFaceH4/ultrachat_200k`). You can replace this with your own dataset. If your dataset is in a local file (e.g., CSV, JSON), you would load it using `datasets.load_dataset` with the appropriate path and format.

After loading, we'll tokenize the dataset.

In [ ]:
print('Attempting to fix pyarrow incompatibility error by reinstalling pyarrow and pandas...')
!pip install --upgrade --force-reinstall pyarrow pandas
print('Reinstallation complete. Please re-run the previous cell (2. Load and Prepare the Dataset) after this cell finishes execution.')

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd # Import pandas for DataFrame operations

# Assuming df already contains the uploaded train.csv data from the previous cell.
# If df is not globally available or needs to be reloaded:
# df = pd.read_csv('train.csv') # Uncomment and run if df is not available

# Convert the pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Reduce training samples to 5000 as requested
dataset = dataset.shuffle(seed=42).select(range(5000))

print(f"Loaded dataset with {len(dataset)} samples.")
print("First sample:", dataset[0])

# Define a formatting function for the custom dataset (train.csv)
# It expects 'instruction', 'output', and optionally 'input' columns.
def formatting_function_for_csv(example):
    instruction = example.get('instruction', '')
    output = example.get('output', '')
    input_text = example.get('input', '')

    # Check if 'input' is empty or NaN and format accordingly
    if pd.isna(input_text) or str(input_text).strip() == '':
        formatted_text = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    else:
        formatted_text = f"<|im_start|>user\n{instruction}\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    return {"text": formatted_text}

# Apply the new formatting function
formatted_dataset = dataset.map(formatting_function_for_csv, remove_columns=dataset.column_names)

# Tokenize the dataset (this function remains the same as it expects a 'text' field)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length") # Reduced max_length

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

print("Tokenized dataset sample:", tokenized_dataset[0])

### 3. Configure LoRA

Now, we'll set up the LoRA configuration using `PeftConfig` from the `peft` library.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Removed: SFTTrainer will call prepare_model_for_kbit_training internally when peft_config is provided.
# model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

# Removed: The SFTTrainer will apply the peft_config directly.
# model = get_peft_model(model, lora_config)

print("LoRA configuration prepared successfully. Model is ready for SFTTrainer.")
# model.print_trainable_parameters() # This will now cause an error as model is not a PeftModel yet

### 4. Set up Training Arguments and Trainer

Finally, we'll define the `TrainingArguments` and use the `SFTTrainer` (from `trl`) for supervised fine-tuning. This setup will be ready for you to run the training.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1, # Further reducing batch size to avoid OOM
    gradient_accumulation_steps=8, # Increasing accumulation steps to maintain effective batch size
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=4,
    lr_scheduler_type="constant",
    report_to="none",
    logging_steps=10,
    save_steps=100,
)

# SFT Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=lora_config,
    args=training_args,
)

print("Trainer initialized. You can now start training by running `trainer.train()`")

In [ ]:
import os

# Set environment variable to help with CUDA memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Start the QLoRA training
trainer.train()

In [ ]:
from google.colab import files
import pandas as pd

print("Please select the file(s) you wish to upload:")
uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

  # Example: If the uploaded file is a CSV, you can read it into a DataFrame
  # Adjust this part if your file is not a CSV or requires different parsing.
  if filename.endswith('.csv'):
    try:
      df = pd.read_csv(filename)
      print(f'Successfully loaded "{filename}" into a pandas DataFrame. First 5 rows:')
      display(df.head())
    except Exception as e:
      print(f'Error reading {filename} as CSV: {e}')
  elif filename.endswith(('.json', '.jsonl')):
      try:
          df = pd.read_json(filename, lines=filename.endswith('.jsonl'))
          print(f'Successfully loaded "{filename}" into a pandas DataFrame. First 5 rows:')
          display(df.head())
      except Exception as e:
          print(f'Error reading {filename} as JSON: {e}')
  else:
    print(f'File "{filename}" uploaded. You can now access its content.')

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes trl pandas


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct" # Target 1.5B parameters model
DATASET_PATH = "train.csv"
OUTPUT_DIR = "./qwen_qlora_output"

# Speed Hyperparameters Optimized for T4 GPU
MAX_SEQ_LENGTH = 128    # Shorter context speeds up computations
BATCH_SIZE = 8          # Larger batch fully saturates T4 GPU
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4
MAX_STEPS = 100         # Hard-limit training to exactly 100 steps (~1-2 minutes on T4)
SAMPLE_LIMIT = 2000     # Limit dataset size for quick preprocessing

device_map = "auto"
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------
# Load and Preprocess Dataset
# ---------------------------------------------------------------------
print(f"Loading dataset from {DATASET_PATH}...")
dataset = load_dataset("csv", data_files=DATASET_PATH)
dataset = dataset["train"]

# Select subset
if SAMPLE_LIMIT and len(dataset) > SAMPLE_LIMIT:
    print(f"Selecting first {SAMPLE_LIMIT} samples from dataset...")
    dataset = dataset.select(range(SAMPLE_LIMIT))

# Split 90/10 (just to get a train split)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]

print(split_dataset)

# ---------------------------------------------------------------------
# Load Tokenizer & Apply Chat Template
# ---------------------------------------------------------------------
print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def format_prompts(example):
    text_list = []
    for inst, inp, out in zip(example["instruction"], example["input"], example["output"]):
        user_content = inst
        if inp and str(inp).strip() and str(inp).strip().lower() != "nan":
            user_content = f"{inst}\n\nInput Context:\n{inp}"

        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": out}
        ]

        formatted = tokenizer.apply_chat_template(messages, tokenize=False)
        text_list.append(formatted)

    return {"text": text_list}

print("Formatting dataset prompts...")
train_dataset = train_dataset.map(format_prompts, batched=True)

# ---------------------------------------------------------------------
# Load Model with 4-bit Quantization (QLoRA)
# ---------------------------------------------------------------------
print(f"Loading base model: {MODEL_NAME} in 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map=device_map,
    attn_implementation="sdpa", # Native PyTorch SDPA attention
    dtype=torch.float16 # Changed from torch_dtype to dtype
)

model = prepare_model_for_kbit_training(model)

# ---------------------------------------------------------------------
# Setup LoRA Configuration
# ---------------------------------------------------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Reduced targets for faster training
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("LoRA Adapter loaded successfully!")
model.print_trainable_parameters()

# ---------------------------------------------------------------------
# Setup Trainer
# ---------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,      # Capping steps is the fastest way to train
    logging_steps=10,
    optim="paged_adamw_8bit",
    fp16=False,               # Explicitly disable fp16 to avoid BFloat16 error
    bf16=False,               # Explicitly disable bfloat16
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    # max_seq_length=MAX_SEQ_LENGTH, # Removed as it causes TypeError
    # packing=True,             # Removed as it causes TypeError
    args=training_args,
)

# ---------------------------------------------------------------------
# Run Fine-Tuning
# ---------------------------------------------------------------------
print("\nStarting QLoRA Fine-Tuning...\n")
trainer.train()

# ---------------------------------------------------------------------
# Save LoRA Adapters
# ---------------------------------------------------------------------
SAVE_PATH = "qwen_qlora_dolly"
print(f"Saving fine-tuned adapters to {SAVE_PATH}...")
trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model saved successfully!")

### 5. Inference and Comparison: Base Model vs. Fine-Tuned Model

Now that the model is fine-tuned, let's compare its performance against the original base model using a sample prompt.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- Configuration for Inference ---
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FINE_TUNED_MODEL_PATH = "./qwen_qlora_dolly"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load Base Model and Tokenizer ---
print(f"Loading base model: {BASE_MODEL_NAME}...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16, # Use bfloat16 for base model for general compatibility
    device_map="auto"
)
base_model.eval() # Set base model to evaluation mode
print("Base model loaded successfully!")

# --- Load Fine-Tuned Model and Tokenizer ---
print(f"Loading fine-tuned model from: {FINE_TUNED_MODEL_PATH}...")
# The tokenizer is saved with the adapter
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED_MODEL_PATH, trust_remote_code=True)
fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token

# Load the base model again (or use the one loaded above) and then attach the LoRA adapters
# For 4-bit, we need to re-load the base model with the BitsAndBytesConfig if it wasn't quantized initially
# or load the adapter directly if the base was already quantized.

# If the base model was loaded with BitsAndBytesConfig, we can just load the adapter on top.
# Since our base model was loaded in 4-bit (implicitly), we can load the PeftModel on top.
fine_tuned_model = PeftModel.from_pretrained(base_model, FINE_TUNED_MODEL_PATH)
fine_tuned_model = fine_tuned_model.merge_and_unload() # Merge LoRA layers for inference
fine_tuned_model.eval() # Set fine-tuned model to evaluation mode
print("Fine-tuned model loaded and merged successfully!")

In [ ]:
print("Upgrading torchao to a compatible version...")
!pip install --upgrade torchao
print("torchao upgrade complete. Please re-run the inference cells (from 'Loading base model' onwards).")

In [ ]:
# --- Define a Sample Prompt ---
PROMPT = "What is the capital of France?"

# Function to generate response
def generate_response(model, tokenizer, prompt_text):
    messages = [
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(DEVICE)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )
    generated_ids = [output_ids[len(model_inputs["input_ids"][0]):] for output_ids in generated_ids]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# --- Generate Response with Base Model ---
print("\n--- Generating with Base Model ---")
base_response = generate_response(base_model, base_tokenizer, PROMPT)
print(f"Base Model Response: {base_response}")

# --- Generate Response with Fine-Tuned Model ---
print("\n--- Generating with Fine-Tuned Model ---")
fine_tuned_response = generate_response(fine_tuned_model, fine_tuned_tokenizer, PROMPT)
print(f"Fine-Tuned Model Response: {fine_tuned_response}")

print("\nComparison Complete!")

In [ ]:
from google.colab import files
import os

# Path where the model was saved
SAVE_PATH = "qwen_qlora_dolly"

# List of files to download
files_to_download = [
    os.path.join(SAVE_PATH, "adapter_model.safetensors"),
    os.path.join(SAVE_PATH, "adapter_config.json"),
    os.path.join(SAVE_PATH, "tokenizer_config.json"),
    os.path.join(SAVE_PATH, "tokenizer.json"),
    os.path.join(SAVE_PATH, "chat_template.jinja"),
    os.path.join(SAVE_PATH, "README.md") # Include README if it exists
]

print(f"Attempting to download files from '{SAVE_PATH}'...")

for file_path in files_to_download:
    if os.path.exists(file_path):
        try:
            files.download(file_path)
            print(f"Successfully downloaded: {file_path}")
        except Exception as e:
            print(f"Error downloading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

print("Download process complete.")